# RentRadar — SQL Analysis

Analytical queries over the normalized PostgreSQL schema (`cities → localities → listings`).
Connects via `DATABASE_URL` from `.env`. Run `db/load_to_postgres.py` first.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

### 1. Average rent by city

In [2]:
# Average rent by city
pd.read_sql("""
    SELECT c.name AS city, ROUND(AVG(li.rent)) AS avg_rent, COUNT(*) AS n_listings
    FROM listings li
    JOIN localities l ON li.locality_id = l.locality_id
    JOIN cities c     ON l.city_id = c.city_id
    GROUP BY c.name
    ORDER BY avg_rent DESC;
""", engine)

,city,avg_rent,n_listings
0,Mumbai,73376.0,1619
1,Bangalore,50397.0,1763
2,New Delhi,35468.0,1788
3,Pune,30816.0,1773
4,Nagpur,18016.0,595


### 2. Furnished premium per city

In [3]:
# Furnished premium per city
pd.read_sql("""
    SELECT c.name AS city,
           ROUND(AVG(li.rent) FILTER (WHERE li.furnishing = 'Furnished'))   AS furnished_avg,
           ROUND(AVG(li.rent) FILTER (WHERE li.furnishing = 'Unfurnished')) AS unfurnished_avg
    FROM listings li
    JOIN localities l ON li.locality_id = l.locality_id
    JOIN cities c     ON l.city_id = c.city_id
    GROUP BY c.name
    ORDER BY furnished_avg DESC;
""", engine)

,city,furnished_avg,unfurnished_avg
0,Mumbai,82115.0,63435.0
1,Bangalore,63900.0,40611.0
2,Pune,44616.0,23849.0
3,New Delhi,37786.0,30574.0
4,Nagpur,28484.0,13508.0


### 3. Top 10 priciest localities (reliable sample)

In [4]:
# Top 10 priciest localities (>= 10 listings)
pd.read_sql("""
    SELECT c.name AS city, l.name AS locality,
           ROUND(AVG(li.rent)) AS avg_rent, COUNT(*) AS n_listings
    FROM listings li
    JOIN localities l ON li.locality_id = l.locality_id
    JOIN cities c     ON l.city_id = c.city_id
    GROUP BY c.name, l.name
    HAVING COUNT(*) >= 10
    ORDER BY avg_rent DESC
    LIMIT 10;
""", engine)

,city,locality,avg_rent,n_listings
0,New Delhi,Defence Colony,173941.0,17
1,Mumbai,Worli,156850.0,20
2,Mumbai,Prabhadevi,144500.0,14
3,Mumbai,Bandra West,142035.0,17
4,New Delhi,GK II,141667.0,12
5,Bangalore,Indira Nagar,129500.0,10
6,Mumbai,Lower Parel,124400.0,10
7,Bangalore,Hebbal,119857.0,35
8,Mumbai,Juhu,118333.0,12
9,Mumbai,Parel,114792.0,24


### 4. Price-per-sqft leaders per city

In [5]:
# Price-per-sqft leaders within each city (top 3)
pd.read_sql("""
    SELECT city, locality, avg_rate, city_rank FROM (
        SELECT c.name AS city, l.name AS locality,
               ROUND(AVG(li.area_rate), 2) AS avg_rate,
               RANK() OVER (PARTITION BY c.name ORDER BY AVG(li.area_rate) DESC) AS city_rank
        FROM listings li
        JOIN localities l ON li.locality_id = l.locality_id
        JOIN cities c     ON l.city_id = c.city_id
        GROUP BY c.name, l.name
        HAVING COUNT(*) >= 10
    ) ranked
    WHERE city_rank <= 3
    ORDER BY city, city_rank;
""", engine)

,city,locality,avg_rate,city_rank
0,Bangalore,Marathahalli,116.19,1
1,Bangalore,Rajaji Nagar,58.73,2
2,Bangalore,KR Puram,56.20,3
3,Mumbai,Worli,166.30,1
4,Mumbai,Bandra West,155.65,2
5,Mumbai,Juhu,135.58,3
6,Nagpur,Mihan,20.40,1
7,Nagpur,Manish Nagar,19.33,2
8,Nagpur,Trimurti Nagar,19.18,3
9,New Delhi,GK II,114.18,1
